# OpenTelemetry (.NET) — Raw Execution Trace Extraction

Analyze **C#/.NET repositories** with **OpenTelemetry (.NET)** and **OTLP/Jaeger/Zipkin exporters**.

This notebook preserves the original OpenTelemetry trace output exactly as emitted during execution.

**Default benchmark repository:** [csharp-testing-opentelemetry](https://github.com/visvantha-testable/csharp-testing-opentelemetry)

> **Note:** This workflow extracts raw trace fields only. It does **not** calculate taxonomy metrics or custom scores.


## Section 1 — Install Dependencies

Install Python packages, verify `.NET SDK`, `dotnet tool list`, and OpenTelemetry packages in the repository.


In [ ]:
!pip install -q gitpython pandas jupyter
!pip install -q -r requirements.txt


In [ ]:
import os
import sys
from pathlib import Path

os.environ.pop('PYTHONPATH', None)
METRIC_ROOT = Path('.').resolve()
if not (METRIC_ROOT / 'tool' / '_opentelemetry_utils.py').exists():
    METRIC_ROOT = Path('..').resolve()
TOOL_ROOT = METRIC_ROOT / 'tool'
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from _opentelemetry_utils import (
    ANALYSIS_TYPE, PROGRAMMING_LANGUAGE, REPO_URL, TOOL_NAME, NotebookLogger,
    collect_prerequisite_versions, download_dotnet_sdk, dotnet_env, ensure_output_dirs, resolve_metric_root,
)

METRIC_ROOT = resolve_metric_root(METRIC_ROOT)
DIRS = ensure_output_dirs(METRIC_ROOT)
OUTPUT_DIR = DIRS['outputs']
WORKSPACE_DIR = DIRS['workspace']
LOGGER = NotebookLogger(OUTPUT_DIR / 'error_log.txt')

DOTNET_ROOT = download_dotnet_sdk(DIRS['runtimes'], tmp_dir=DIRS['tmp'])
DOTNET_ENV = dotnet_env(DOTNET_ROOT, tmp_dir=DIRS['tmp'])
print(f'.NET SDK root: {DOTNET_ROOT}')


## Section 2 — Configuration

Choose Git clone mode or local repository mode. Select the telemetry exporter format to preserve.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/visvantha-testable/csharp-testing-opentelemetry.git'

LOCAL_REPO_PATH = './workspace/csharp-testing-opentelemetry'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

EXPORTER = 'otlp'  # supported: otlp | jaeger | zipkin

IF_CLONE_EXISTS = 'reuse'  # reuse | reclone

# Local mode example:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/csharp-testing-opentelemetry'


## Section 3 — Repository Setup

Clone or reuse the repository and display repository metadata.


In [ ]:
from IPython.display import display

from _opentelemetry_utils import (
    compute_repository_stats, discover_csharp_inventory, discover_solution, resolve_repository_path,
)

OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

try:
    REPO_PATH, CLONE_STATUS = resolve_repository_path(
        USE_GIT_URL, REPO_URL, LOCAL_REPO_PATH, WORKSPACE_PATH, IF_CLONE_EXISTS, LOGGER
    )
except Exception as exc:
    LOGGER.error(f'Repository setup failed: {exc}')
    raise

INVENTORY_DF = discover_csharp_inventory(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, INVENTORY_DF)
SOLUTION_PATH = discover_solution(REPO_PATH)

print(CLONE_STATUS)
print(f"Repository Name: {REPO_STATS['repository_name']}")
print(f"Repository Size (C# bytes): {REPO_STATS['repository_size_bytes']:,}")
print(f"Number of C# Files: {REPO_STATS['csharp_file_count']}")
print(f"Current Commit Hash: {REPO_STATS['commit_hash']}")
print(f"Solution File: {SOLUTION_PATH}")
print('Project Files:')
for project in REPO_STATS['project_files']:
    print(f'  - {project}')


## Section 4 — Discover Source Files

Recursively discover `*.cs`, `*.csproj`, and `*.sln` files and export the inventory CSV.


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'csharp_inventory.csv'
INVENTORY_DF.to_csv(INVENTORY_CSV, index=False)
print(f'Discovered {len(INVENTORY_DF)} source/build files')
print(f'Saved inventory to: {INVENTORY_CSV}')
display(INVENTORY_DF.head(20))


## Section 5 — Restore Dependencies

Run `dotnet restore` and capture stdout, stderr, exit code, and execution time.


In [ ]:
from _opentelemetry_utils import run_dotnet_restore

try:
    RESTORE_RESULT = run_dotnet_restore(REPO_PATH, SOLUTION_PATH, DOTNET_ROOT, DOTNET_ENV)
    print('--- restore stdout ---')
    print(RESTORE_RESULT['stdout'])
    print('--- restore stderr ---')
    print(RESTORE_RESULT['stderr'])
    print(f"Exit Code: {RESTORE_RESULT['returncode']}")
    print(f"Execution Time (ms): {RESTORE_RESULT['elapsed_ms']}")
    (OUTPUT_PATH / 'restore_stdout.txt').write_text(RESTORE_RESULT.get('stdout', ''), encoding='utf-8')
    (OUTPUT_PATH / 'restore_stderr.txt').write_text(RESTORE_RESULT.get('stderr', ''), encoding='utf-8')
    if not RESTORE_RESULT['success']:
        LOGGER.error('dotnet restore failed.', file=RESTORE_RESULT.get('target', 'restore'))
except Exception as exc:
    RESTORE_RESULT = {'success': False, 'stdout': '', 'stderr': str(exc), 'returncode': 1, 'elapsed_ms': 0}
    LOGGER.error(f'Restore step raised an exception: {exc}')


## Section 6 — Build Project

Run `dotnet build` and capture stdout, stderr, exit code, and execution time.


In [ ]:
from _opentelemetry_utils import run_dotnet_build

BUILD_RESULT = {'success': False, 'stdout': '', 'stderr': 'Skipped because restore failed.', 'returncode': 1, 'elapsed_ms': 0}
if RESTORE_RESULT.get('success'):
    try:
        BUILD_RESULT = run_dotnet_build(REPO_PATH, SOLUTION_PATH, DOTNET_ROOT, DOTNET_ENV)
        print('--- build stdout ---')
        print(BUILD_RESULT['stdout'])
        print('--- build stderr ---')
        print(BUILD_RESULT['stderr'])
        print(f"Exit Code: {BUILD_RESULT['returncode']}")
        print(f"Execution Time (ms): {BUILD_RESULT['elapsed_ms']}")
        if not BUILD_RESULT['success']:
            LOGGER.error('dotnet build failed.', file=BUILD_RESULT.get('target', 'build'))
    except Exception as exc:
        BUILD_RESULT = {'success': False, 'stdout': '', 'stderr': str(exc), 'returncode': 1, 'elapsed_ms': 0}
        LOGGER.error(f'Build step raised an exception: {exc}')


## Section 7 — Execute Repository

Run the executable project (`dotnet run`) and capture stdout, stderr, execution time, and exit code.


In [ ]:
from _opentelemetry_utils import discover_executable_project, run_dotnet_execute

RUN_RESULT = {'success': False, 'stdout': '', 'stderr': 'Skipped because build failed.', 'returncode': 1, 'elapsed_ms': 0}
EXECUTABLE_PROJECT = None
if BUILD_RESULT.get('success'):
    EXECUTABLE_PROJECT = discover_executable_project(REPO_PATH, LOGGER)
    if EXECUTABLE_PROJECT is None:
        LOGGER.error('No executable project detected.', file=str(REPO_PATH))
    else:
        try:
            RUN_RESULT = run_dotnet_execute(REPO_PATH, EXECUTABLE_PROJECT, DOTNET_ROOT, DOTNET_ENV)
            print(f"Executable Project: {RUN_RESULT.get('project')}")
            print('--- run stdout ---')
            print(RUN_RESULT['stdout'])
            print('--- run stderr ---')
            print(RUN_RESULT['stderr'])
            print(f"Exit Code: {RUN_RESULT['returncode']}")
            print(f"Execution Time (ms): {RUN_RESULT['elapsed_ms']}")
            if not RUN_RESULT['success']:
                LOGGER.error('dotnet run failed.', file=RUN_RESULT.get('project', 'run'))
        except Exception as exc:
            RUN_RESULT = {'success': False, 'stdout': '', 'stderr': str(exc), 'returncode': 1, 'elapsed_ms': 0}
            LOGGER.error(f'Run step raised an exception: {exc}')


## Section 8 — Collect OpenTelemetry Output

Detect exported telemetry for the selected exporter (`otlp`, `jaeger`, or `zipkin`) and preserve the original JSON payload.


In [ ]:
from _opentelemetry_utils import collect_environment_json, find_exporter_file

RAW_SOURCE = find_exporter_file(REPO_PATH, EXPORTER)
ENVIRONMENT = collect_environment_json(DOTNET_ROOT, DOTNET_ENV, EXPORTER)
PREREQ_DF = collect_prerequisite_versions(DOTNET_ROOT, DOTNET_ENV, REPO_PATH)
display(PREREQ_DF)

if RAW_SOURCE is None:
    LOGGER.error(f'No telemetry export detected for exporter={EXPORTER}', file=str(REPO_PATH))
    print('Telemetry export not found. Check artifacts/training or configured collector endpoints.')
else:
    print(f'Detected telemetry source: {RAW_SOURCE}')


## Section 9 — Extract Raw Trace Output

Copy the original exporter JSON verbatim to `outputs/opentelemetry_raw_output.json` without modification.

Span fields available in the preserved payload:

```text
TraceId, SpanId, ParentSpanId, OperationName, SpanName, StartTime, EndTime,
Duration, Status, Events, Attributes, ResourceAttributes, InstrumentationScope,
ExceptionEvents, Links
```


In [ ]:
import json

from _opentelemetry_utils import extract_spans_from_payload, preserve_raw_trace_output

RAW_OUTPUT_PATH = OUTPUT_PATH / 'opentelemetry_raw_output.json'
SPANS = []
if RAW_SOURCE is not None:
    preserve_raw_trace_output(RAW_SOURCE, RAW_OUTPUT_PATH)
    try:
        PAYLOAD = json.loads(RAW_SOURCE.read_text(encoding='utf-8'))
        SPANS = extract_spans_from_payload(PAYLOAD)
        print(f'Preserved raw trace output at: {RAW_OUTPUT_PATH}')
        print(f'Extracted span records available for CSV parsing: {len(SPANS)}')
    except json.JSONDecodeError as exc:
        LOGGER.error(f'Malformed telemetry JSON: {exc}', file=str(RAW_SOURCE))
else:
    RAW_OUTPUT_PATH.write_text('{}', encoding='utf-8')
    LOGGER.error('Raw trace output unavailable; wrote empty JSON placeholder.', file=str(RAW_OUTPUT_PATH))


## Section 10 — Parse Raw Output

Generate `spans.csv`, `events.csv`, and `attributes.csv` from the preserved trace payload.


In [ ]:
from _opentelemetry_utils import build_attributes_dataframe, build_events_dataframe, build_spans_dataframe

SPANS_DF = build_spans_dataframe(SPANS)
EVENTS_DF = build_events_dataframe(SPANS)
ATTRIBUTES_DF = build_attributes_dataframe(SPANS)

SPANS_CSV = OUTPUT_PATH / 'spans.csv'
EVENTS_CSV = OUTPUT_PATH / 'events.csv'
ATTRIBUTES_CSV = OUTPUT_PATH / 'attributes.csv'
SPANS_DF.to_csv(SPANS_CSV, index=False)
EVENTS_DF.to_csv(EVENTS_CSV, index=False)
ATTRIBUTES_DF.to_csv(ATTRIBUTES_CSV, index=False)

print(f'Spans CSV: {SPANS_CSV}')
print(f'Events CSV: {EVENTS_CSV}')
print(f'Attributes CSV: {ATTRIBUTES_CSV}')
display(SPANS_DF.head(20))


## Section 11 — Execution Summary

Display trace counts and overall execution status.


In [ ]:
import pandas as pd

from _opentelemetry_utils import build_execution_summary, read_text

EXECUTION_STATUS = 'SUCCESS' if RUN_RESULT.get('success') and SPANS else ('PARTIAL' if SPANS else 'FAILED')
SUMMARY = build_execution_summary(SPANS, EXECUTION_STATUS)

summary_df = pd.DataFrame(
    [
        {'Metric': 'Total Traces', 'Value': SUMMARY['total_traces']},
        {'Metric': 'Total Spans', 'Value': SUMMARY['total_spans']},
        {'Metric': 'Root Spans', 'Value': SUMMARY['root_spans']},
        {'Metric': 'Child Spans', 'Value': SUMMARY['child_spans']},
        {'Metric': 'Exception Events', 'Value': SUMMARY['exception_events']},
        {'Metric': 'Execution Status', 'Value': SUMMARY['execution_status']},
    ]
)
display(summary_df)

EXECUTION_METADATA = {
    'clone_status': CLONE_STATUS,
    'repository_stats': REPO_STATS,
    'restore': RESTORE_RESULT,
    'build': BUILD_RESULT,
    'run': RUN_RESULT,
    'telemetry_source': str(RAW_SOURCE) if RAW_SOURCE else '',
    'exporter': EXPORTER,
    'summary': SUMMARY,
}
(OUTPUT_PATH / 'execution_metadata.json').write_text(json.dumps(EXECUTION_METADATA, indent=2, default=str), encoding='utf-8')
(OUTPUT_PATH / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')
(OUTPUT_PATH / 'build_stdout.txt').write_text(BUILD_RESULT.get('stdout', ''), encoding='utf-8')
(OUTPUT_PATH / 'build_stderr.txt').write_text(BUILD_RESULT.get('stderr', ''), encoding='utf-8')
(OUTPUT_PATH / 'run_stdout.txt').write_text(RUN_RESULT.get('stdout', ''), encoding='utf-8')
(OUTPUT_PATH / 'run_stderr.txt').write_text(RUN_RESULT.get('stderr', ''), encoding='utf-8')


## Section 12 — Error Handling

Review captured errors and confirm deliverables under `outputs/`.


In [ ]:
LOGGER.write_errors()
ERROR_LOG = OUTPUT_PATH / 'error_log.txt'
print(f'Error log: {ERROR_LOG}')
print(read_text(ERROR_LOG) or 'No errors recorded.')

deliverables = [
    'opentelemetry_raw_output.json',
    'spans.csv',
    'events.csv',
    'attributes.csv',
    'csharp_inventory.csv',
    'execution_metadata.json',
    'environment.json',
    'build_stdout.txt',
    'build_stderr.txt',
    'run_stdout.txt',
    'run_stderr.txt',
    'error_log.txt',
]
print('\nDeliverables:')
for name in deliverables:
    path = OUTPUT_PATH / name
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {path}')
